# Stage 5B — Label the wider dataset and split by time

**What:** Read the exact 12-day dataset from Stage 5A, create 72-hour labels, and make three groups: training (January–August), validation (September–October), and test (November–December).

**Why:** We train on earlier requests and evaluate on later ones to simulate the future. We keep the test group untouched until the model choice is finished.

The 12 monthly days are still a learning sample. They let us practice honest time-based evaluation; later we will expand the scheduled collection.

In [ ]:
# Cell 2 — Why: choose one exact dataset manifest.
from google.colab import drive
from pathlib import Path
import hashlib, json, pandas as pd

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
dataset_id = "20260926T112608997328Z"
manifest_path = root / "datasets" / dataset_id / "manifest.json"
dataset = json.loads(manifest_path.read_text())
if dataset["dataset_id"] != dataset_id or len(dataset["partitions"]) != 12:
    raise ValueError("Wrong or incomplete dataset manifest")
print("Selected dataset:", dataset_id)

In [ ]:
# Cell 3 — Why: verify every source file before combining its rows.
records = []
for part in dataset["partitions"]:
    raw = (root / part["file"]).read_bytes()
    if hashlib.sha256(raw).hexdigest() != part["sha256"]:
        raise ValueError(f"Changed file for {part['day']}")
    day_rows = [json.loads(line) for line in raw.splitlines()]
    if len(day_rows) != part["rows"]:
        raise ValueError(f"Wrong row count for {part['day']}")
    if any(not row["created_date"].startswith(part["day"]) for row in day_rows):
        raise ValueError(f"Wrong date in the {part['day']} partition")
    records.extend(day_rows)

df = pd.DataFrame(records).reindex(
    columns=["unique_key", "created_date", "closed_date", "complaint_type", "borough"]
)
if len(df) != dataset["total_rows"] or df["unique_key"].duplicated().any():
    raise ValueError("Combined rows are incomplete or duplicated")
print("Verified rows from 12 days:", len(df))

In [ ]:
# Cell 4 — Why: compute the same 72-hour answer for all 12 days.
created_local = pd.to_datetime(df["created_date"], errors="coerce")
closed_local = pd.to_datetime(df["closed_date"], errors="coerce")
created = created_local.dt.tz_localize(
    "America/New_York", ambiguous="NaT", nonexistent="NaT"
).dt.tz_convert("UTC")
closed = closed_local.dt.tz_localize(
    "America/New_York", ambiguous="NaT", nonexistent="NaT"
).dt.tz_convert("UTC")
as_of = pd.Timestamp(dataset["created_at_utc"])
deadline = created + pd.Timedelta(hours=72)

bad = created.isna() | (df["closed_date"].notna() & closed.isna())
bad |= closed.notna() & ((closed < created) | (closed > as_of))
if (deadline[~bad] > as_of).any():
    raise ValueError("Some requests are too recent to label")
labels = (closed.isna() | (closed > deadline)).astype(int)
print("Invalid times excluded:", int(bad.sum()))

In [ ]:
# Cell 5 — Why: use only creation-time inputs and hold out later months.
usable = df.loc[~bad, [
    "unique_key", "created_date", "complaint_type", "borough"
]].copy()
usable["month"] = created_local.loc[~bad].dt.month.to_numpy()
usable["hour"] = created_local.loc[~bad].dt.hour.to_numpy()
usable["weekday"] = created_local.loc[~bad].dt.weekday.to_numpy()
usable["label"] = labels.loc[~bad].to_numpy()

train = usable[usable["month"] <= 8].copy()
valid = usable[usable["month"].isin([9, 10])].copy()
test = usable[usable["month"] >= 11].copy()
if min(len(train), len(valid), len(test)) == 0:
    raise ValueError("A time split has no rows")
print("Train:", len(train), "Validation:", len(valid), "Test:", len(test))

In [ ]:
# Cell 6 — Why: save the split and its exact parent dataset ID.
from datetime import datetime, timezone

split_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
split_dir = root / "splits" / split_id
split_dir.mkdir(parents=True, exist_ok=False)
files = {}
for name, frame in {"train": train, "validation": valid, "test": test}.items():
    clean = frame.astype(object).where(pd.notna(frame), None)
    content = "".join(
        json.dumps(row, sort_keys=True, allow_nan=False) + "\n"
        for row in clean.to_dict(orient="records")
    ).encode("utf-8")
    split_file = split_dir / f"{name}.jsonl"
    split_file.write_bytes(content)
    if hashlib.sha256(split_file.read_bytes()).hexdigest() != hashlib.sha256(content).hexdigest():
        raise ValueError(f"{name}: saved file failed verification")
    files[name] = {"rows": len(frame), "sha256": hashlib.sha256(content).hexdigest()}

split_manifest = {"split_id": split_id, "source_dataset_id": dataset_id,
                  "rule": "train Jan-Aug; validation Sep-Oct; test Nov-Dec 2025",
                  "label_rule": "1 if not closed within 72 actual hours; else 0",
                  "time_zone_assumption": "America/New_York",
                  "invalid_times_excluded": int(bad.sum()),
                  "files": files, "lesson_version": "stage5b-v1"}
(split_dir / "manifest.json").write_text(
    json.dumps(split_manifest, indent=2), encoding="utf-8"
)
print("Split ID:", split_id)
print("Saved under:", split_dir)

# Stage 5B takeaway and quiz

**Why this stage mattered:** We verified every raw partition, applied one label rule, kept only information known at request creation, and saved a chronological split with its parent dataset ID. The test set is reserved for one final evaluation after choosing the model.

**Easy quiz:**

1. Why do we train on earlier months and test on later months?
2. What is the validation group for?
3. Why is `closed_date` absent from the saved split files?
4. Why do we record the parent dataset ID?

Send me `Verified rows from 12 days`, `Invalid times excluded`, the train/validation/test counts, `Split ID`, and your answers. Stage 5C will train a baseline and a first model.